[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alextfkd/TorchCode/blob/master/solutions/54_sgd_momentum_solution.ipynb)

# ✅ Solution: SGD with Momentum

**SGD with Momentum** を実装する。誰もが最初に学ぶ optimizer、今でも CNN (ResNet, VGG) では Adam と競合する。

> 💡 **どこで使う**：CV (ResNet, VGG, ViT 含む) で今も第一選択の optimizer。Adam より汎化性能が良いことが多い。

<details>
<summary>📖 もっと詳しく</summary>

更新式は PyTorch 慣習で `v = μv + g; p -= lr * v`（`(1-μ)` factor なし）。

Nesterov 版もある (`nesterov=True`)。Cosine LR (#30) と組み合わせる ResNet レシピが定番。Transformer 系は AdamW (#56) が標準。

</details>

### Update rule (PyTorch 規約 — `(1-μ)` factor は **ない**)
$$v_{t+1} = \mu \cdot v_t + g_t$$
$$p_{t+1} = p_t - \text{lr} \cdot v_{t+1}$$

注意: 古典的な Heavy-Ball / EMA 形式は `μ*v + (1-μ)*g`。`torch.optim.SGD` の規約は **`(1-μ)` factor がない**。matching test を通すため PyTorch 規約に合わせる。


### Signature
```python
class MySGDMomentum:
    def __init__(self, params, lr, momentum=0.9): ...
    def zero_grad(self): ...
    def step(self): ...
```

### Rules
- `torch.optim.SGD` は **使わない**
- `step()` と `zero_grad()` の 2 メソッド実装
- Velocity update: `v = momentum * v + p.grad`（PyTorch 規約 — `(1-μ)` factor なし）
- Param update: `p -= lr * v`
- Velocity buffer を zero で初期化（param 1個あたり 1個）
- `step()` を `@torch.no_grad()` で wrap（または手動 context）

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/alextfkd/TorchCode.git')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

class MySGDMomentum:
    def __init__(self, params, lr, momentum=0.9):
        self.params = list(params)
        self.lr = lr
        self.momentum = momentum
        self.velocities = [torch.zeros_like(p) for p in self.params]
    
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.detach_()
                p.grad.zero_()
    
    @torch.no_grad()
    def step(self):
        for p, v in zip(self.params, self.velocities):
            if p.grad is None:
                continue
            v.mul_(self.momentum).add_(p.grad)
            p.add_(v, alpha=-self.lr)

In [ ]:
import torch
torch.manual_seed(0)
p = torch.randn(5, requires_grad=True)
opt = MySGDMomentum([p], lr=0.1, momentum=0.9)
for step in range(3):
    loss = (p ** 2).sum()
    loss.backward()
    opt.step()
    opt.zero_grad()
    print(f'step {step}: |p| = {p.norm().item():.4f}')

In [ ]:
from torch_judge import check
check("sgd_momentum")